# Hypothesis Testing: Earthquake Impact on Housing Sales
### DSA210 Course Project — Umay Aslan

---
This notebook implements **statistical hypothesis tests** to evaluate whether
small-to-medium earthquakes (M4.0–6.0) have a short-term impact on housing sales
in selected Turkish provinces.

We will focus on:
- Selecting an example province (e.g. Istanbul, Izmir, Balikesir, Ankara)
- Choosing a specific earthquake event from AFAD data
- Creating **pre-shock** and **post-shock** time windows (e.g. 3 months before vs 3 months after)
- Running statistical tests (two-sample t-test as a starting point)
- Interpreting p-values in the context of the project hypotheses (H0 / H1)

---


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

housing = pd.read_excel("../KONUT-DORT-SEHIR.xls")
earthquakes = pd.read_excel("../DEPREM-DORT-SEHIR.xlsx")

housing.head(), earthquakes.head()

## Check Column Names
Before building the analysis, we inspect the columns to adapt the code
to the actual dataset structure.


In [ ]:
print("Housing columns:\n", housing.columns.tolist())
print("\nEarthquake columns:\n", earthquakes.columns.tolist())

 **IMPORTANT**

In the next cells, you must replace the placeholder column names
with the actual names from your datasets. Typical structures might be:

- Housing data: `['Tarih', 'Il', 'Toplam_Satis']` or similar
- Earthquake data: `['Tarih', 'Il', 'Buyukluk']` or similar

Update the code below accordingly.


## Select Province & Earthquake Event
We choose one province (e.g., **Istanbul**) and one earthquake event
to build a pre/post comparison window.


In [ ]:
# EDIT THESE NAMES BASED ON YOUR DATA
PROVINCE_COLUMN_H = "Il"          # housing province column
DATE_COLUMN_H = "Tarih"           # housing date column
SALES_COLUMN = "Toplam_Satis"     # housing sales column

PROVINCE_COLUMN_E = "Il"          # earthquake province column
DATE_COLUMN_E = "Tarih"           # earthquake date column
MAG_COLUMN = "Buyukluk"           # earthquake magnitude column

# Choose a province for analysis
province = "Istanbul"  # example; change to Ankara / Izmir / Balikesir as needed

# Filter data for selected province
housing_p = housing[housing[PROVINCE_COLUMN_H] == province].copy()
eq_p = earthquakes[earthquakes[PROVINCE_COLUMN_E] == province].copy()

# Convert date columns to datetime
housing_p[DATE_COLUMN_H] = pd.to_datetime(housing_p[DATE_COLUMN_H])
eq_p[DATE_COLUMN_E] = pd.to_datetime(eq_p[DATE_COLUMN_E])

# Sort by date
housing_p.sort_values(DATE_COLUMN_H, inplace=True)
eq_p.sort_values(DATE_COLUMN_E, inplace=True)

housing_p.head(), eq_p.head()

### Choose a Specific Earthquake (Example: Largest Magnitude Event)
Here we pick the **largest magnitude** earthquake in the selected province
during the time range as the main event.


In [ ]:
# Select the largest magnitude earthquake for the province
main_eq = eq_p.sort_values(MAG_COLUMN, ascending=False).iloc[0]
event_date = main_eq[DATE_COLUMN_E]
event_mag = main_eq[MAG_COLUMN]

print("Selected event province:", province)
print("Event date:", event_date)
print("Magnitude:", event_mag)

## Define Pre- and Post-Event Windows
We build a time window around the selected earthquake date.
For example:
- 3 months **before** the event
- 3 months **after** the event

You may later experiment with different window sizes (1 month, 6 months, etc.).


In [ ]:
window_months = 3

pre_start = event_date - pd.DateOffset(months=window_months)
pre_end = event_date  # strictly before the event

post_start = event_date
post_end = event_date + pd.DateOffset(months=window_months)

print("Pre window:", pre_start.date(), "to", pre_end.date())
print("Post window:", post_start.date(), "to", post_end.date())

# Filter housing sales into pre and post windows
mask_pre = (housing_p[DATE_COLUMN_H] >= pre_start) & (housing_p[DATE_COLUMN_H] < pre_end)
mask_post = (housing_p[DATE_COLUMN_H] >= post_start) & (housing_p[DATE_COLUMN_H] <= post_end)

sales_pre = housing_p.loc[mask_pre, SALES_COLUMN]
sales_post = housing_p.loc[mask_post, SALES_COLUMN]

print("Number of pre observations:", len(sales_pre))
print("Number of post observations:", len(sales_post))

## Visual Comparison (Optional but Recommended)
We illustrate the difference between pre- and post-event periods.


In [ ]:
plt.figure(figsize=(10,4))
plt.plot(housing_p[DATE_COLUMN_H], housing_p[SALES_COLUMN], marker="o")
plt.axvline(event_date, linestyle="--")
plt.title(f"Housing Sales in {province} with Earthquake Event Marked")
plt.xlabel("Date")
plt.ylabel("Housing Sales")
plt.tight_layout()
plt.show()

## Statistical Test (Two-Sample t-test)

**Null Hypothesis (H0):**
> Mean housing sales in the pre-event period = mean housing sales in the post-event period.

**Alternative Hypothesis (H1):**
> Mean housing sales are different before and after the earthquake (two-sided test).


In [ ]:
# Drop any missing values just in case
sales_pre_clean = sales_pre.dropna()
sales_post_clean = sales_post.dropna()

t_stat, p_value = stats.ttest_ind(sales_pre_clean, sales_post_clean, equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_value)

alpha = 0.05
if p_value < alpha:
    print("\nResult: Reject H0 at 5% significance level. There is evidence of a change in housing sales.")
else:
    print("\nResult: Fail to reject H0 at 5% significance level. No strong evidence of change in housing sales.")

## Interpretation Template (for Report)

You can adapt the following text to your final report:

> For province **X**, using a time window of **3 months** before and after the earthquake
> on **[date]** with magnitude **M**, we conducted a two-sample t-test on monthly housing sales.
> The resulting p-value was **p = ...**. At a 5% significance level, we **(reject / fail to reject)**
> the null hypothesis that mean housing sales are equal before and after the event. This suggests that
> the earthquake **(did / did not)** have a statistically significant short-term impact on housing sales
> in **province X**.

You can repeat this analysis for multiple provinces and events to build a comparative discussion section.
